# YOLOv8s Sample-Aware Box/DFL Weighting on Kaggle

This notebook runs the `YOLOv8s` baseline structure with the sample-aware box/DFL weighting experiment enabled.

Design choices in this notebook:
- backbone / neck / head unchanged
- cls loss unchanged (`BCE`)
- only box loss and DFL loss are reweighted
- optimizer is fixed to `AdamW`
- seed is fixed to `42`


In [ ]:
# Cell 1: clone latest repo
%cd /kaggle/working
!rm -rf yolov8s_kuangjing
!git clone https://github.com/tuanziya666/yolov8s_kuangjing.git
%cd /kaggle/working/yolov8s_kuangjing
!git pull
!python -m pip uninstall -y ultralytics >/dev/null 2>&1 || true
!python -m pip install -q -e .
!python -c "import ultralytics, sys; print('python=', sys.executable); print('ultralytics=', ultralytics.__file__)"

In [ ]:
# Cell 2: create dataset yaml
# Change DATASET_ROOT to your Kaggle dataset root
DATASET_ROOT = "/kaggle/input/your-dataset-root"

from pathlib import Path

yaml_text = f"""path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: chuck
  1: gripper
  2: drill_pipe
  3: coal_miner
"""

cfg_path = Path('/kaggle/working/yolov8s_kuangjing/ultralytics/cfg/datasets/coalmine4_kaggle.yaml')
cfg_path.write_text(yaml_text, encoding='utf-8')
print(cfg_path.read_text(encoding='utf-8'))

In [ ]:
# Cell 3: fix AMP check assets
import requests
from pathlib import Path
from PIL import Image

assets_dir = Path('/kaggle/working/yolov8s_kuangjing/ultralytics/assets')
assets_dir.mkdir(parents=True, exist_ok=True)

def ensure_valid_image(path: Path, url: str) -> None:
    valid = False
    if path.exists():
        try:
            with Image.open(path) as im:
                im.verify()
            valid = True
        except Exception:
            path.unlink(missing_ok=True)
    if not valid:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        with Image.open(path) as im:
            im.verify()

ensure_valid_image(assets_dir / 'bus.jpg', 'https://ultralytics.com/images/bus.jpg')
ensure_valid_image(assets_dir / 'zidane.jpg', 'https://ultralytics.com/images/zidane.jpg')
print('assets fixed')

In [ ]:
# Cell 4: common config
import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TRAIN_DEVICE = '0'      # If dual GPU is stable for you on Kaggle, you can try '0,1'
EPOCHS = 300
BATCH = 16
IMGSZ = 640
WORKERS = 2
RUN_PROJECT = '/kaggle/working/runs'
RUN_NAME = 'yolov8s_saboxdfl_adamw_seed42_300ep'

# Sample-aware weighting hyperparameters
SA_BOX_ENABLE = True
SA_SMALL_ALPHA = 0.5
SA_ELONG_BETA = 0.7
SA_CLASS_GAMMA = 0.5
SA_SMALL_AREA_THR = 0.012521
SA_ELONG_RATIO_THR = 3.0
SA_TARGET_CLASS_IDS = '2'

print('TRAIN_DEVICE =', TRAIN_DEVICE)
print('EPOCHS =', EPOCHS)
print('BATCH =', BATCH)
print('RUN_NAME =', RUN_NAME)
print('SA_TARGET_CLASS_IDS =', SA_TARGET_CLASS_IDS)

In [ ]:
# Cell 5: train
%cd /kaggle/working/yolov8s_kuangjing

import os
from ultralytics import YOLO

os.environ['ULTRALYTICS_CLS_LOSS'] = 'bce'
os.environ['ULTRALYTICS_IOU_LOSS'] = 'ciou'
os.environ.pop('ULTRALYTICS_INNER_IOU_RATIO', None)
os.environ.pop('ULTRALYTICS_WCIOU_AC_LAMBDA', None)
os.environ.pop('ULTRALYTICS_WCIOU_AC_GAMMA', None)

os.environ['ULTRALYTICS_SA_BOX_ENABLE'] = 'true' if SA_BOX_ENABLE else 'false'
os.environ['ULTRALYTICS_SA_SMALL_ALPHA'] = str(SA_SMALL_ALPHA)
os.environ['ULTRALYTICS_SA_ELONG_BETA'] = str(SA_ELONG_BETA)
os.environ['ULTRALYTICS_SA_CLASS_GAMMA'] = str(SA_CLASS_GAMMA)
os.environ['ULTRALYTICS_SA_SMALL_AREA_THR'] = str(SA_SMALL_AREA_THR)
os.environ['ULTRALYTICS_SA_ELONG_RATIO_THR'] = str(SA_ELONG_RATIO_THR)
os.environ['ULTRALYTICS_SA_TARGET_CLASS_IDS'] = SA_TARGET_CLASS_IDS

model = YOLO('yolov8s.pt')
model.train(
    data='ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    task='detect',
    project=RUN_PROJECT,
    name=RUN_NAME,
    device=TRAIN_DEVICE,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    workers=WORKERS,
    seed=SEED,
    deterministic=True,
    pretrained=True,
    optimizer='AdamW',
    cache=False,
    amp=True,
    save=True,
    val=True,
    plots=True,
    verbose=True,
    exist_ok=False,
)


In [ ]:
# Cell 6: check outputs
from pathlib import Path

run_dir = Path(RUN_PROJECT) / RUN_NAME
print('Run directory:', run_dir)
print('results.csv exists:', (run_dir / 'results.csv').exists())
print('best.pt exists:', (run_dir / 'weights' / 'best.pt').exists())
print('last.pt exists:', (run_dir / 'weights' / 'last.pt').exists())
if (run_dir / 'results.csv').exists():
    import pandas as pd
    df = pd.read_csv(run_dir / 'results.csv')
    cols = [c for c in df.columns if c.startswith('sa/')]
    print('sample-aware columns:', cols)
    display(df[cols + ['epoch']].tail() if cols else df.tail())